# 좌표 정규화 및 구도 분류 검증

## 개요

본 노트북은 인테리어 이미지 상품 태그의 좌표 정규화 과정을 수행하고,  
촬영 구도에 따라 좌표 정규화 방식을 달리해야 하는지를 통계적으로 검증합니다.

## 분석 흐름

1. 데이터 로드 및 불필요 컬럼 제거
2. 좌표 정규화 (0~1 스케일링, Grid Zone)
3. 구도 분류 검증
   - 수동 라벨링 결과 분포 확인
   - 샘플 대표성 검증 (place_label 분포 비교, Cramér's V)
   - 천장샷 비율 안정성 검증 (샘플 크기별)
   - 구도별 태그 좌표 분포 통계 검증 (t-test)
4. 결론
5. 최종 데이터셋 저장

## 1. 라이브러리 임포트

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chisquare
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import rc

rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

라이브러리 로드 완료


## 2. 데이터 로드 및 컬럼 정리

원본 데이터셋(194만건)에서 아래 기준으로 불필요한 컬럼을 제거합니다.

| 제거 기준 | 해당 컬럼 |
|---|---|
| 완전 중복 | `place` (place_value와 동일), `additional_category_text_3` (2와 100% 동일) |
| 전체 결측 | `manual_category` |
| 중간 파생 컬럼 | `category_text`, `category_final`, `category_by_productId`, `category_final_simple` (최종 검수본 `category_final_reviewed`만 유지) |
| 중복 description | `desc_prefix`, `description_clean` (원본 `description` 유지) |

In [2]:
df = pd.read_csv('housewarming_image_products_category_place_final.csv')
print(f'원본 shape: {df.shape}')

drop_cols = [
    'place',
    'place_value',
    'additional_category_text_2',
    'additional_category_text_3',
    'manual_category',
    'category_final_simple',
    'category_by_productId',
    'category_final',
    'category_text',
    'desc_prefix',
    'description_clean',
]

df = df.drop(columns=drop_cols)

print(f'정리 후 shape: {df.shape}')
print(f'컬럼 수: 38 → {len(df.columns)}')
print(df.columns.tolist())

원본 shape: (1941343, 38)
정리 후 shape: (1941343, 27)
컬럼 수: 38 → 27
['contentId', 'postTitle', 'postUrl', 'contentIndex', 'cardId', 'imageUrl', 'imageWidth', 'imageHeight', 'tagIndex', 'positionX', 'positionY', 'description', 'onHide', 'isLikely', 'productId', 'brand', 'productName', 'productUrl', 'production_id', 'production_brand_id', 'production_brand_name', 'production_name', 'originalImageUrl', 'originalPrice', 'sellingPrice', 'category_final_reviewed', 'place_label']


## 3. 좌표 정규화

상품 태그 좌표(`positionX`, `positionY`)를 두 가지 방식으로 정규화합니다.

### 3-1. 0~1 정규화
- 실제 합성 시 좌표 리매핑 용도
- `positionX / 100`, `positionY / 100`

### 3-2. Grid Zone (3×3)
- 배치 패턴 매칭 용도
- 이미지를 3×3 구역으로 나눠 각 태그의 위치를 구역명으로 표현

```
┌────────┬────────┬────────┐
│ 좌상단 │  상단  │ 우상단 │
├────────┼────────┼────────┤
│  좌측  │  중앙  │  우측  │
├────────┼────────┼────────┤
│ 좌하단 │  하단  │ 우하단 │
└────────┴────────┴────────┘
```

In [3]:
# 0~1 정규화
df['positionX_norm'] = df['positionX'] / 100
df['positionY_norm'] = df['positionY'] / 100

# Grid Zone (3×3)
def get_grid_zone(x, y):
    col = min(int(x / 33.34), 2)
    row = min(int(y / 33.34), 2)
    zone_map = {
        (0,0): '좌상단', (1,0): '상단',   (2,0): '우상단',
        (0,1): '좌측',   (1,1): '중앙',   (2,1): '우측',
        (0,2): '좌하단', (1,2): '하단',   (2,2): '우하단',
    }
    return zone_map[(col, row)]

df['grid_zone'] = df.apply(lambda r: get_grid_zone(r['positionX'], r['positionY']), axis=1)

print('좌표 정규화 완료')
print(df[['positionX', 'positionY', 'positionX_norm', 'positionY_norm', 'grid_zone']].head(10))

좌표 정규화 완료
   positionX  positionY  positionX_norm  positionY_norm grid_zone
0    8.47374    2.95458        0.084737        0.029546       좌상단
1   33.40250   22.25110        0.334025        0.222511        상단
2   35.91370   68.12710        0.359137        0.681271        하단
3   92.59730   58.03110        0.925973        0.580311        우측
4    2.63499    2.70093        0.026350        0.027009       좌상단
5   50.21690   38.98830        0.502169        0.389883        중앙
6   13.05120   68.89320        0.130512        0.688932       좌하단
7    2.20120   79.13130        0.022012        0.791313       좌하단
8   24.79910   88.16950        0.247991        0.881695       좌하단
9   31.94440   57.11030        0.319444        0.571103        좌측


## 4. 구도 분류 검증

### 연구 질문
> 촬영 구도(정면샷, 사선샷, 천장샷 등)에 따라 좌표 정규화 방식을 달리해야 하는가?

### 검증 전략

1. **수동 라벨링 결과 분포**: 1,000장을 직접 라벨링하여 구도 분포 확인
2. **샘플 대표성 검증**: 샘플이 전체 데이터를 잘 대표하는지 통계적으로 확인
3. **천장샷 비율 안정성**: 샘플 크기를 늘려도 비율이 수렴하는지 확인
4. **태그 좌표 분포 t-test**: 구도 간 좌표 분포에 통계적 차이가 있는지 확인

### 4-1. 수동 라벨링 결과 분포

1,000장을 직접 라벨링한 결과를 확인합니다.

> **라벨링 도구**: `labeling.py` (별도 실행 파일)  
> **분류 기준**: 일반샷(정면/사선), 클로즈업, 천장샷

In [4]:
# 라벨링 결과 로드
labels = pd.read_csv('view_labels.csv')
labels_sample = pd.read_csv('labeling_sample.csv')

print('=== 1000장 수동 라벨링 결과 ===')
print(labels['view_type'].value_counts())
print()
print('=== 비율 (%) ===')
print((labels['view_type'].value_counts(normalize=True) * 100).round(2))

=== 1000장 수동 라벨링 결과 ===
view_type
정면      791
클로즈업    191
천장샷       2
Name: count, dtype: int64

=== 비율 (%) ===
view_type
정면      80.39
클로즈업    19.41
천장샷      0.20
Name: proportion, dtype: float64


### 4-2. 샘플 대표성 검증

카이제곱 적합도 검정으로 샘플의 `place_label` 분포가 전체 데이터와 통계적으로 유사한지 확인합니다.

- p < 0.05이더라도 **효과 크기(Cramér's V)**가 0.1 미만이면 실질적 차이는 미미함
- Cramér's V < 0.1 → 대표성 양호 / 0.1~0.3 → 중간 / 0.3 이상 → 대표성 낮음

In [11]:
# place_label 분포 비교
total_place = df['place_label'].value_counts(normalize=True).round(3) * 100
sample_place = labels_sample.merge(df[['cardId','place_label']].drop_duplicates(), on='cardId')['place_label'].value_counts(normalize=True).round(3) * 100

print("=== place_label 분포 비교 ===")
print(pd.DataFrame({'전체': total_place, '샘플': sample_place}))

=== place_label 분포 비교 ===
               전체    샘플
place_label            
가구&소품         3.9   3.6
거실           27.0  24.6
도면 및 가구배치도    0.0   NaN
드레스룸          2.7   3.2
베란다           2.0   2.7
복도            1.6   2.1
비포사진          0.2   0.4
사무공간          0.5   0.5
상업공간          0.0   NaN
서재&작업실        7.4   7.4
시공 리뷰         0.1   0.2
아이방           4.0   2.8
외관&기타         1.4   2.3
욕실            5.6   8.2
원룸            2.4   2.2
자기소개          0.3   0.7
전문가리뷰         0.0   NaN
전문가소개         0.0   0.1
제품후기          0.4   0.8
주방           22.2  19.6
침실           14.7  14.5
현관            3.3   4.1


In [14]:
# 전체 분포 비율
total_dist = df['place_label'].value_counts(normalize=True)

# 샘플 분포
sample_place = labels_sample.merge(
    df[['cardId','place_label']].drop_duplicates(), on='cardId'
)['place_label'].value_counts()

# 공통 카테고리만
common = sample_place.index
expected_ratio = total_dist[common] / total_dist[common].sum()  # 공통 카테고리만 재정규화
expected = expected_ratio * sample_place.sum()  # 샘플 합계에 맞춤

chi2, p = chisquare(f_obs=sample_place[common], f_exp=expected)

print('=== 샘플 대표성 카이제곱 적합도 검정 ===')
print(f'chi2: {chi2:.4f}')
print(f'p-value: {p:.4f}')
print(f'→ {"샘플이 전체를 대표하지 않음 (p<0.05)" if p < 0.05 else "샘플이 전체를 통계적으로 대표함 (p>0.05)"}')

=== 샘플 대표성 카이제곱 적합도 검정 ===
chi2: 52.7576
p-value: 0.0000
→ 샘플이 전체를 대표하지 않음 (p<0.05)


In [15]:
# Cramér's V (효과 크기)
n = sample_place.sum()
k = len(common)
cramers_v = np.sqrt(chi2 / (n * (k - 1)))

print(f"Cramér's V: {cramers_v:.4f}")
print(f"→ {'작은 효과 (대표성 양호)' if cramers_v < 0.1 else '중간 효과' if cramers_v < 0.3 else '큰 효과 (대표성 낮음)'}")

Cramér's V: 0.0541
→ 작은 효과 (대표성 양호)


### 4-3. 천장샷 비율 안정성 검증

샘플 크기를 달리하여도 천장샷 비율이 일정하게 유지되는지 확인합니다.  
비율이 수렴한다면 샘플 편향이 아닌 **실제 데이터 특성**임을 의미합니다.

In [12]:
# 샘플 크기별 천장샷 비율 확인
sample_sizes = [200, 400, 600, 800, 1000]
ceiling_ratios = []

for size in sample_sizes:
    subset = labels.head(size)
    ratio = (subset['view_type'] == '천장샷').sum() / size * 100
    ceiling_ratios.append(ratio)
    print(f"{size}장 기준 천장샷 비율: {ratio:.2f}%")

# 시각화
plt.figure(figsize=(8, 5))
plt.plot(sample_sizes, ceiling_ratios, marker='o', color='steelblue', linewidth=2)
plt.axhline(y=ceiling_ratios[-1], color='tomato', linestyle='--', alpha=0.7, label=f'최종 비율 {ceiling_ratios[-1]:.2f}%')
plt.xlabel('샘플 크기')
plt.ylabel('천장샷 비율 (%)')
plt.title('샘플 크기별 천장샷 비율 안정성')
plt.legend()
plt.tight_layout()
plt.savefig('ceiling_ratio_stability.png', dpi=150)
plt.show()

200장 기준 천장샷 비율: 0.50%
400장 기준 천장샷 비율: 0.25%
600장 기준 천장샷 비율: 0.17%
800장 기준 천장샷 비율: 0.25%
1000장 기준 천장샷 비율: 0.20%


### 4-4. 구도별 태그 좌표 분포 통계 검증

일반샷과 클로즈업 간 태그 좌표 분산을 t-test로 비교합니다.

- **귀무가설**: 구도에 따라 태그 좌표 분포에 유의미한 차이가 없다 (p > 0.05)

In [5]:
# 라벨링 데이터와 원본 데이터 merge
merged = df.merge(labels, on='cardId')

# 천장샷 제외 (0.2%로 통계적으로 무의미)
merged = merged[merged['view_type'] != '천장샷']

normal = merged[merged['view_type'] == '정면']
closeup = merged[merged['view_type'] == '클로즈업']

print(f'일반샷: {len(normal)}개')
print(f'클로즈업: {len(closeup)}개')

일반샷: 4800개
클로즈업: 758개


In [6]:
# 시각화
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('구도별 태그 좌표 분포 비교', fontsize=14)

# positionX 분포
axes[0,0].hist(normal['positionX'], bins=30, alpha=0.7, label='일반샷', color='steelblue')
axes[0,0].hist(closeup['positionX'], bins=30, alpha=0.7, label='클로즈업', color='tomato')
axes[0,0].set_title('positionX 분포 비교')
axes[0,0].legend()

# positionY 분포
axes[0,1].hist(normal['positionY'], bins=30, alpha=0.7, label='일반샷', color='steelblue')
axes[0,1].hist(closeup['positionY'], bins=30, alpha=0.7, label='클로즈업', color='tomato')
axes[0,1].set_title('positionY 분포 비교')
axes[0,1].legend()

# 카드당 태그 수
normal_tag_count = normal.groupby('cardId').size()
closeup_tag_count = closeup.groupby('cardId').size()
axes[1,0].hist(normal_tag_count, bins=20, alpha=0.7, label='일반샷', color='steelblue')
axes[1,0].hist(closeup_tag_count, bins=20, alpha=0.7, label='클로즈업', color='tomato')
axes[1,0].set_title('카드당 태그 수 비교')
axes[1,0].legend()

# 태그 분산 (공간 퍼짐)
normal_spread = normal.groupby('cardId').apply(
    lambda x: np.sqrt(x['positionX'].var() + x['positionY'].var())
).dropna()
closeup_spread = closeup.groupby('cardId').apply(
    lambda x: np.sqrt(x['positionX'].var() + x['positionY'].var())
).dropna()
axes[1,1].hist(normal_spread, bins=20, alpha=0.7, label='일반샷', color='steelblue')
axes[1,1].hist(closeup_spread, bins=20, alpha=0.7, label='클로즈업', color='tomato')
axes[1,1].set_title('태그 분산(공간 퍼짐) 비교')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('coordinate_analysis.png', dpi=150)
plt.show()

In [7]:
# 태그 좌표 분산 t-test
t1, p1 = stats.ttest_ind(normal_spread, closeup_spread)
print('=== 태그 좌표 분산 t-test ===')
print(f't-statistic: {t1:.4f}')
print(f'p-value: {p1:.4f}')
print(f'→ {"유의미한 차이 있음 (p<0.05)" if p1 < 0.05 else "유의미한 차이 없음 (p>0.05)"}')

=== 태그 좌표 분산 t-test ===
t-statistic: -0.7752
p-value: 0.4384
→ 유의미한 차이 없음 (p>0.05)


## 5. 결론

### 검증 결과 요약

| 검증 항목 | 결과 | 해석 |
|---|---|---|
| 수동 라벨링 천장샷 비율 | 0.20% | 전체의 0.2%에 불과 |
| 샘플 대표성 (Cramér's V) | 0.054 | 0.1 미만 → 실질적 차이 미미, 대표성 양호 |
| 천장샷 비율 안정성 | 0.2~0.5% 수렴 | 샘플 크기와 무관하게 안정적 |
| 태그 좌표 분산 t-test | p = 0.4384 | 구도 간 유의미한 차이 없음 |

### 결론

1. 1,000장 수동 라벨링 결과 **천장샷은 전체의 0.2%** 에 불과하며, 샘플 크기를 늘려도 이 비율은 0.2~0.5% 범위에서 안정적으로 유지됨
2. 샘플의 공간 유형(place_label) 분포는 전체 데이터와 Cramér's V = 0.054로 **실질적 차이가 미미**하여 샘플이 전체를 잘 대표함
3. 일반샷과 클로즈업 간 **태그 좌표 분산 t-test 결과 p = 0.44** 로 통계적으로 유의미한 차이가 없음
4. 따라서 구도 분류에 따른 좌표 정규화 방식 차별화는 불필요하며, **단일 0~1 정규화**를 전체 데이터에 적용하는 것이 통계적으로 타당함을 검증

### 한계점 및 향후 개선 방향

- 본 검증은 1,000장 샘플 기준으로 진행되어 전체 34만 카드를 대표하기에 한계가 있을 수 있음
- 향후 개선 방향으로 **Depth Estimation 기반 3D 좌표 변환**을 고려할 수 있음  
  (예: MiDaS 모델을 활용한 깊이 추정 → 구도와 무관한 공간 좌표 추출)

## 6. 최종 데이터셋 저장

In [8]:
df.to_csv('housewarming_cleaned.csv', index=False, encoding='utf-8-sig')

print(f'최종 저장 완료 → housewarming_cleaned.csv')
print(f'shape: {df.shape}')
print(f'컬럼: {df.columns.tolist()}')

최종 저장 완료 → housewarming_cleaned.csv
shape: (1941343, 30)
컬럼: ['contentId', 'postTitle', 'postUrl', 'contentIndex', 'cardId', 'imageUrl', 'imageWidth', 'imageHeight', 'tagIndex', 'positionX', 'positionY', 'description', 'onHide', 'isLikely', 'productId', 'brand', 'productName', 'productUrl', 'production_id', 'production_brand_id', 'production_brand_name', 'production_name', 'originalImageUrl', 'originalPrice', 'sellingPrice', 'category_final_reviewed', 'place_label', 'positionX_norm', 'positionY_norm', 'grid_zone']
